# GenerativeAURORA — Baseline F: Monte Carlo Dropout LSTM

### Purpose
This notebook is a faithful conversion of the supplied `step12_Baseline_F_MCDropOut.py` script.

The baseline uses **Monte Carlo Dropout**: dropout remains active at inference time and repeated stochastic forward passes form the scenario ensemble.

### Architecture
The supplied implementation uses:
- a 2-layer bidirectional LSTM;
- regime and site embeddings;
- dropout probability `0.3`;
- 50 stochastic forward passes to create 50 scenario samples.

### Workflow position
This is an uncertainty-estimation baseline used alongside the other alternative models for comparison with GenerativeAURORA.

### Faithful-conversion rule
The attached `.py` file is the source of truth. The executable Python below is copied **verbatim** and only divided into logical notebook cells.

No architecture, hyperparameter, training rule, metric, path, filename, or inference procedure is silently changed. Markdown cells are added only to explain the original implementation.


## 1. Header and imports

This opening block is preserved verbatim from the supplied script. It describes Monte Carlo Dropout as the probabilistic baseline and explains its intended uncertainty-sampling mechanism.

In [ ]:
# GenerativeAURORA - Baseline F: Monte Carlo Dropout LSTM
#
# MC Dropout approximates Bayesian uncertainty by keeping dropout active
# at inference time and sampling multiple forward passes as scenarios.
# This is a widely used probabilistic baseline in energy forecasting papers.
#
# Known weakness: underestimates uncertainty under distribution shift
# (e.g. rare Overcast days not seen during training) -- exactly where
# GenerativeAURORA + RCCC excels.
#
# Architecture:
#   2-layer BiLSTM with dropout=0.3 (kept ON at inference)
#   Regime + site embeddings (same as GenerativeAURORA)
#   Output: point prediction per forward pass
#   Scenarios: 50 stochastic forward passes = 50 scenario samples
#
# Run:
#   python baseline_f_mc_dropout.py

import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Paths

Defines the original dataset, evaluation, and MC Dropout checkpoint directories. The Windows paths are intentionally preserved.

In [ ]:
# -----------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------
DATASET_DIR = "/content/drive/MyDrive/Colab Notebooks/Dataset"
EVAL_DIR    = os.path.join(DATASET_DIR, "evaluation")
CKPT_DIR    = os.path.join(DATASET_DIR, "checkpoints_mcdropout")
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

## 3. Hyperparameters and labels

Preserves the original horizon, lookback, feature count, embedding and hidden dimensions, dropout probability, training settings, gradient clipping, and number of stochastic forward passes.

In [ ]:
# -----------------------------------------------------------------------
# HYPERPARAMETERS
# -----------------------------------------------------------------------
HORIZON     = 16
LOOKBACK    = 16
N_FEATURES  = 10
EMBED_DIM   = 32
HIDDEN_DIM  = 128
DROPOUT_P   = 0.3     # kept active at inference
BATCH_SIZE  = 128
EPOCHS      = 150
LR          = 1e-3
WEIGHT_DECAY= 1e-5
GRAD_CLIP   = 1.0
N_SCENARIOS = 50      # = number of MC forward passes

REGIME_NAMES = {0: "Clear", 1: "Cloudy", 2: "Overcast"}
SITE_NAMES   = {0: "Phoenix_AZ", 1: "LosAngeles_CA", 2: "Denver_CO",
                3: "Miami_FL",   4: "Seattle_WA"}

## 4. Dataset and weighted sampler

Loads the sequence arrays and creates the same inverse-frequency regime sampler used by the supplied baseline.

In [ ]:
# -----------------------------------------------------------------------
# DATASET
# -----------------------------------------------------------------------
class SolarSequenceDataset(Dataset):
    def __init__(self, npz_path):
        data        = np.load(npz_path)
        self.X      = torch.tensor(data["X"],      dtype=torch.float32)
        self.Y      = torch.tensor(data["Y"],      dtype=torch.float32)
        self.regime = torch.tensor(data["regime"], dtype=torch.long)
        self.site   = torch.tensor(data["site"],   dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.regime[idx], self.site[idx]

def make_weighted_sampler(dataset):
    labels = dataset.regime.numpy()
    unique, counts = np.unique(labels, return_counts=True)
    freq    = dict(zip(unique.tolist(), counts.tolist()))
    weights = np.array([1.0 / freq[int(r)] for r in labels], dtype=np.float32)
    return WeightedRandomSampler(torch.from_numpy(weights), len(weights), replacement=True)

## 5. MC Dropout LSTM model

Defines the 2-layer bidirectional LSTM with regime/site embeddings and explicit dropout layers. The source script keeps the model in training mode during scenario generation so dropout remains active.

In [ ]:
# -----------------------------------------------------------------------
# MODEL: MC Dropout LSTM
# -----------------------------------------------------------------------
class MCDropoutLSTM(nn.Module):
    def __init__(self, n_features=N_FEATURES, lookback=LOOKBACK,
                 horizon=HORIZON, hidden=HIDDEN_DIM, embed_dim=EMBED_DIM,
                 dropout_p=DROPOUT_P, n_regimes=3, n_sites=5):
        super().__init__()
        self.dropout_p  = dropout_p
        self.regime_emb = nn.Embedding(n_regimes, embed_dim)
        self.site_emb   = nn.Embedding(n_sites,   embed_dim)

        # BiLSTM -- note: nn.LSTM dropout only applies between layers,
        # so we add explicit dropout layers for MC Dropout at output
        self.lstm = nn.LSTM(
            input_size  = n_features + embed_dim * 2,
            hidden_size = hidden,
            num_layers  = 2,
            batch_first = True,
            bidirectional = True,
            dropout     = dropout_p,   # between LSTM layers
        )
        self.dropout = nn.Dropout(p=dropout_p)
        self.norm    = nn.LayerNorm(hidden * 2)
        self.head    = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),   # MC Dropout: also in head
            nn.Linear(hidden, horizon),
            nn.Sigmoid(),
        )

    def forward(self, x, regime, site):
        B, L, F = x.shape
        r_e = self.regime_emb(regime).unsqueeze(1).expand(-1, L, -1)
        s_e = self.site_emb(site).unsqueeze(1).expand(-1, L, -1)
        inp = torch.cat([x, r_e, s_e], dim=-1)
        out, _ = self.lstm(inp)
        h      = self.dropout(self.norm(out.mean(1)))   # MC Dropout here
        return self.head(h)   # (B, H)

    def mc_sample(self, x, regime, site, n_scenarios=N_SCENARIOS):
        """
        Keep model in TRAIN mode to activate dropout.
        Run n_scenarios forward passes to get scenario ensemble.
        """
        self.train()   # CRITICAL: keeps dropout active
        scenarios = []
        with torch.no_grad():
            for _ in range(n_scenarios):
                pred = self.forward(x, regime, site)   # (B, H)
                scenarios.append(pred.unsqueeze(1))     # (B, 1, H)
        return torch.cat(scenarios, dim=1)              # (B, S, H)

## 6. Metrics

Implements the original W/m² metrics: CRPS, pinball loss, 80%/90% coverage and interval width, median MAE, and the printed metric summary.

In [ ]:
# -----------------------------------------------------------------------
# METRICS
# -----------------------------------------------------------------------
def load_scalers():
    with open(os.path.join(DATASET_DIR, "scaler_params.json"), "r") as f:
        return json.load(f)

def inv_scale_ghi(arr, scalers):
    s = scalers["GHI"]
    return arr * (s["max"] - s["min"]) + s["min"]

def crps_score(actuals, scenarios):
    N, S, H = scenarios.shape
    t1 = np.abs(scenarios - actuals[:, np.newaxis, :]).mean(axis=1)
    t2 = np.zeros((N, H))
    for n in range(N):
        sc = scenarios[n]
        t2[n] = np.abs(sc[:, np.newaxis, :] - sc[np.newaxis, :, :]).mean(axis=(0,1))
    return float((t1 - 0.5 * t2).mean())

def pinball_np(actuals, scenarios, q):
    fc = np.quantile(scenarios, q, axis=1)
    e  = actuals - fc
    return float(np.where(e >= 0, q*e, (q-1)*e).mean())

def coverage_width(actuals, scenarios, alpha=0.10):
    lo  = np.quantile(scenarios, alpha/2,   axis=1)
    hi  = np.quantile(scenarios, 1-alpha/2, axis=1)
    cov = float(((actuals >= lo) & (actuals <= hi)).mean()) * 100
    wid = float((hi - lo).mean())
    return round(cov, 4), round(wid, 4)

def med_mae(actuals, scenarios):
    return float(np.abs(np.median(scenarios, axis=1) - actuals).mean())

def print_metrics(actuals, scenarios, label=""):
    crps       = crps_score(actuals, scenarios)
    pb10       = pinball_np(actuals, scenarios, 0.10)
    pb50       = pinball_np(actuals, scenarios, 0.50)
    pb90       = pinball_np(actuals, scenarios, 0.90)
    cov90, w90 = coverage_width(actuals, scenarios, 0.10)
    cov80, _   = coverage_width(actuals, scenarios, 0.20)
    mae        = med_mae(actuals, scenarios)
    if label: print("  " + label)
    print("    CRPS         : " + str(round(crps, 4)) + " W/m2")
    print("    Pinball p10  : " + str(round(pb10, 4)) + " W/m2")
    print("    Pinball p50  : " + str(round(pb50, 4)) + " W/m2")
    print("    Pinball p90  : " + str(round(pb90, 4)) + " W/m2")
    print("    Coverage 90% : " + str(cov90) + "%")
    print("    Coverage 80% : " + str(cov80) + "%")
    print("    Width 90% PI : " + str(w90) + " W/m2")
    print("    Median MAE   : " + str(round(mae, 4)) + " W/m2")
    return {"crps": crps, "pinball_p10": pb10, "pinball_p50": pb50,
            "pinball_p90": pb90, "coverage_90": cov90, "coverage_80": cov80,
            "width_90": w90, "median_mae": mae}

## 7. Main setup and evaluation

Runs device selection, dataset loading, loaders, model construction, optimizer/scheduler setup, checkpoint/log paths, and the original MSE training loop. Loads the best checkpoint, performs 50 stochastic MC Dropout forward passes per test batch, inverse-scales the results to W/m², evaluates overall/regime/site metrics, and saves the result files.

In [ ]:
# -----------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------
def main():
    print("GenerativeAURORA - Baseline F: MC Dropout LSTM")
    print("Uncertainty via " + str(N_SCENARIOS) + " stochastic forward passes")
    print("")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device : " + str(device))

    train_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_train.npz"))
    val_ds   = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_val.npz"))
    test_ds  = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_test.npz"))
    print("Train: " + str(len(train_ds)) + "  Val: " + str(len(val_ds)) + "  Test: " + str(len(test_ds)))

    sampler      = make_weighted_sampler(train_ds)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=0, pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=(device.type=="cuda"))
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = MCDropoutLSTM().to(device)
    total_p = sum(p.numel() for p in model.parameters())
    print("Parameters: " + str(total_p))

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_val  = float("inf")
    best_path = os.path.join(CKPT_DIR, "mcdropout_best.pt")
    log_path  = os.path.join(DATASET_DIR, "training_log_mcdropout.csv")

    print("\nTraining for " + str(EPOCHS) + " epochs (MSE loss)...")
    print("-" * 55)

    with open(log_path, "w") as f:
        f.write("epoch,train_loss,val_loss\n")

    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        tr_total = 0.0; n = 0
        for X, Y, regime, site in train_loader:
            X, Y   = X.to(device), Y.to(device)
            regime = regime.to(device); site = site.to(device)
            pred   = model(X, regime, site)
            loss   = F.mse_loss(pred, Y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            tr_total += loss.item(); n += 1
        tr = tr_total / n

        model.eval()
        vl_total = 0.0; n = 0
        with torch.no_grad():
            for X, Y, regime, site in val_loader:
                X, Y   = X.to(device), Y.to(device)
                regime = regime.to(device); site = site.to(device)
                pred   = model(X, regime, site)
                loss   = F.mse_loss(pred, Y)
                vl_total += loss.item(); n += 1
        vl = vl_total / n

        scheduler.step()
        is_best = vl < best_val
        if is_best:
            best_val = vl
            torch.save(model.state_dict(), best_path)

        with open(log_path, "a") as f:
            f.write(str(epoch+1) + "," + str(round(tr,6)) + "," + str(round(vl,6)) + "\n")

        if (epoch+1) % 10 == 0 or is_best:
            marker = "  <-- best" if is_best else ""
            print("Epoch " + str(epoch+1).rjust(3) + "/" + str(EPOCHS)
                  + "  train: " + str(round(tr,6)).ljust(10)
                  + "  val: "   + str(round(vl,6)).ljust(10) + marker)

    elapsed = round((time.time() - t0) / 60, 1)
    print("-" * 55)
    print("Training complete. Time: " + str(elapsed) + " min")
    print("Best val loss: " + str(round(best_val, 6)))
    print("")

    # Evaluation with MC Dropout
    print("Evaluating with " + str(N_SCENARIOS) + " MC Dropout forward passes...")
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    # NOTE: model.train() is called inside mc_sample() to keep dropout active
    scalers = load_scalers()

    all_sc, all_act, all_reg, all_sit = [], [], [], []
    for X, Y, regime, site in test_loader:
        X      = X.to(device)
        regime = regime.to(device); site = site.to(device)
        sc     = model.mc_sample(X, regime, site, N_SCENARIOS).cpu().numpy()
        all_sc.append(sc)
        all_act.append(Y.numpy())
        all_reg.append(regime.cpu().numpy())
        all_sit.append(site.cpu().numpy())

    scenarios = np.concatenate(all_sc,  axis=0)
    actuals   = np.concatenate(all_act, axis=0)
    regimes   = np.concatenate(all_reg, axis=0)
    sites     = np.concatenate(all_sit, axis=0)

    actuals_wm2   = inv_scale_ghi(actuals,                   scalers)
    scenarios_wm2 = inv_scale_ghi(np.clip(scenarios, 0, 1), scalers)

    print("\n=== MC Dropout LSTM: Test Metrics ===")
    overall = print_metrics(actuals_wm2, scenarios_wm2, "Overall (920 days, 5 sites)")

    print("\n=== By Sky Regime ===")
    regime_metrics = {}
    for rid, rname in REGIME_NAMES.items():
        mask = (regimes == rid)
        if mask.sum() == 0: continue
        print("  " + rname + " (n=" + str(mask.sum()) + ")")
        regime_metrics[rname] = print_metrics(actuals_wm2[mask], scenarios_wm2[mask])

    print("\n=== By Site ===")
    site_metrics = {}
    for sid, sname in SITE_NAMES.items():
        mask = (sites == sid)
        if mask.sum() == 0: continue
        print("  " + sname + " (n=" + str(mask.sum()) + ")")
        site_metrics[sname] = print_metrics(actuals_wm2[mask], scenarios_wm2[mask])

    np.savez_compressed(os.path.join(EVAL_DIR, "mcdropout_results.npz"),
                        actuals=actuals_wm2, scenarios=scenarios_wm2,
                        regimes=regimes, sites=sites)
    with open(os.path.join(EVAL_DIR, "mcdropout_metrics.json"), "w") as f:
        json.dump({"model": "MCDropoutLSTM", "overall": overall,
                   "by_regime": regime_metrics, "by_site": site_metrics}, f, indent=2)

    print("\n=== COPY THESE INTO TABLE II ===")
    print("MC Dropout LSTM  |  CRPS: " + str(round(overall["crps"],2))
          + "  | Cov 90%: " + str(overall["coverage_90"])
          + "%  | Width: "  + str(round(overall["width_90"],1))
          + "  | MAE: "     + str(round(overall["median_mae"],2)))
    print("\nResults saved to: " + EVAL_DIR)
    print("All 3 new baselines complete. Run compile_results_table.py to see full table.")

## 8. Script entry point

Preserves the final `if __name__ == "__main__": main()` block exactly.

In [ ]:
if __name__ == "__main__":
    main()

GenerativeAURORA - Baseline F: MC Dropout LSTM
Uncertainty via 50 stochastic forward passes

Device : cpu
Train: 7305  Val: 905  Test: 920
Parameters: 639888

Training for 150 epochs (MSE loss)...
-------------------------------------------------------
Epoch   1/150  train: 0.023386    val: 0.020388    <-- best
Epoch   2/150  train: 0.007373    val: 0.00897     <-- best
Epoch   3/150  train: 0.00538     val: 0.006273    <-- best
Epoch   4/150  train: 0.00443     val: 0.005135    <-- best
Epoch   5/150  train: 0.003724    val: 0.004385    <-- best
Epoch   6/150  train: 0.003338    val: 0.003811    <-- best
Epoch   8/150  train: 0.002743    val: 0.003267    <-- best
Epoch   9/150  train: 0.002606    val: 0.003061    <-- best
Epoch  10/150  train: 0.002391    val: 0.00289     <-- best
Epoch  11/150  train: 0.002212    val: 0.00281     <-- best
Epoch  12/150  train: 0.00217     val: 0.002721    <-- best
Epoch  14/150  train: 0.002025    val: 0.002582    <-- best
Epoch  16/150  train: 0.001

## Execution note

The original Windows paths are intentionally preserved:

`C:/Users/user/Paper_10/Dataset`

For actual Colab execution, this path must point to the corresponding dataset and evaluation locations.

### Main inputs
- `sequences_train.npz`
- `sequences_val.npz`
- `sequences_test.npz`
- `scaler_params.json`

### Main outputs
- `checkpoints_mcdropout/mcdropout_best.pt`
- `training_log_mcdropout.csv`
- `evaluation/mcdropout_results.npz`
- `evaluation/mcdropout_metrics.json`

The original script ends by directing the workflow to `compile_results_table.py`.
